<a href="https://colab.research.google.com/github/jhmoon19/msc-slp/blob/main/ANLP/lab_2_ngrams.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2: Language Identification with character n-gram models



## Character n-gram models: what and why?

In lecture, we talked about n-gram models over **words**, but it's also possible to build n-gram models over **characters**. These can actually be useful for certain tasks, like identifying what language a text is written in.

It's much easier (and requires less data) to build an n-gram model over characters than over words, so will do that here, to help you build your intuitions about this type of model.

You will work with data from three different languages to build and explore character-level n-gram models for a simple language identification task. Along the way, you’ll confront issues like rare characters, smoothing, underflow, and generation strategies.

## What you will learn in this lab

### Tools and practical issues:

In this lab, you will learn:
- how to easily split data into training and development subsets using the `datasets` library.
- how to use these splits, together with a separate test set, to correctly to tune hyperparameters and test generalization.
- some possible pitfalls to watch out for in your data

### Concepts: n-gram models and language identification

After working through the lab, you should be able to:
- compute the probability of a sequence given an n-gram model
- explain how to generate new sentences using an n-gram language model
- explain how to use a character n-gram model to do language identification
   
You should also understand more clearly:
- how n-gram models are trained
- the effects of smoothing and other hyperparameter choices on the model's behaviour (both generation and perplexity)

## 1. Loading and splitting the data

Today you’ll use another dataset we uploaded to Hugging Face. It contains sentences in three languages widely spoken in South Africa:
- **English:** a language in the West Germanic branch of the Indo-European language family with significant influences from Old French. It is spoken as a first language by around 380 million people worldwide, with another billion second-language speakers.
- **Afrikaans:** another language in the West Germanic branch of the Indo-European language family. It evolved from Dutch starting around the 17th century, and is spoken as a first language by around 7 million people, mainly in South Africa and nearby regions.
- **Xhosa: (or isiXhosa):** a language in the Nguni branch of the Bantu language family. It is closely related to Zulu and is spoken as a first language by around 8 million people, mainly in South Africa and nearby regions.

👉 Run the next two cells to load the data.

In [1]:
# once again, we need to update `datasets`
%pip install -U datasets

In [2]:

from datasets import load_dataset

# Our variable names are based on the ISO 639-3 language codes for the languages in the dataset.
# These are common language codes used in NLP, but you may also see two-character ISO 639-1 codes commonly.
# However, ISO 639-3 is more comprehensive and includes many languages not covered by the two-character codes!
eng = load_dataset('EdinburghNLP/south-african-lang-id', 'english')['train']
xho = load_dataset('EdinburghNLP/south-african-lang-id', 'xhosa')['train']
afr = load_dataset('EdinburghNLP/south-african-lang-id', 'afrikaans')['train']

# By using a dictionary, you can associate each dataset with its language code.
# Do you see why we use the code as the key and not the dataset itself?
# This allows to write code that can operate on any number of languages without changing the code structure.
# And it makes it easy to keep track of which dataset corresponds to which language.
# This is a common pattern in NLP and other data processing tasks.
corpora = {'eng': eng, 'xho': xho, 'afr': afr}

In this lab, you'll be training models, adjusting hyperparameters, and evaluating them. To evaluate your changes and avoid over-fitting, you'll use the *train--dev--test* splitting paradigm discussed in class.

You might have noticed that when we loaded the dataset from HuggingFace, we selected a key called `'train'`. That's because you always have to select a split before you can iterate over a HuggingFace dataset, and HuggingFace interprets data without any splits as being a single `'train'` split.

However, we are now going to split this data to create the actual training and development sets, so that you can explore and tune hyperparameters on the development set.

The `datasets` library has a function we can use to easily split off a smaller subset of the data for development, but note that it will call this subset `'test'`!! Despite the name, this split will *not* be used for the test set in this lab. We have set aside another dataset for that, which we will introduce toward the end of the lab.

By default, `train_test_split` selects a *random* subset of the data to split off, which can be useful if there might be differences between the early and late parts of the dataset, and you don't want this to affect your training. However, if you're trying to compare directly to (your own or someone else's) previous work, you need to be careful about whether you split the data the same way.

**Further reading for later:** If you're new to Hugging Face, they provide useful tutorials with much more information about Datasets and other libraries, which you can find [here](https://huggingface.co/docs/datasets/index).

👉 Run the next two cells to load the data and see what its structure is. Then, change the final line in the second cell to print out the first 10 sentences in each language. Do you see anything that surprises you? (You might or might not, remember the data splits are random!)

In [3]:
corpora = {
    lang: dataset.train_test_split(test_size=0.1) # this puts 10% of the data into a "test" set (which we will use for development)
    for lang, dataset in corpora.items() # see how organizing the data in a dictionary allows us to write less code?
}

In [4]:
corpora

{'eng': DatasetDict({
     train: Dataset({
         features: ['text'],
         num_rows: 9000
     })
     test: Dataset({
         features: ['text'],
         num_rows: 1000
     })
 }),
 'xho': DatasetDict({
     train: Dataset({
         features: ['text'],
         num_rows: 9000
     })
     test: Dataset({
         features: ['text'],
         num_rows: 1000
     })
 }),
 'afr': DatasetDict({
     train: Dataset({
         features: ['text'],
         num_rows: 9000
     })
     test: Dataset({
         features: ['text'],
         num_rows: 1000
     })
 })}

In [5]:
# Let's take a quick look at the data.
for lang, dataset in corpora.items():
    print(f'### {lang} data: ###')
    # This line should help you see the structure of the datasets.
    # Replace it with code to print the first ten lines from the training set of each language!
    for i, line in enumerate(dataset['train']['text'][:10], 1):
        print(i, line, sep='\t')
    print()

### eng data: ###
1	The disqualification of a person in terms of subitem - 
2	Each party holding at least 80 seats in the National Assembly is entitled to designate an Executive Deputy President from among the members of the Assembly . 
3	physical address of business :
4	A Municipal Council- 
5	( 3 ) products containing chemicals listed in annexure b may be exported to countries that are not states parties to the chemical weapons convention : provided that the said products contain –
6	or until the person elected President after any election of the National Assembly held before 30 April 1999 , assumes office . 
7	manuals and instructions written or recorded on other media or devices such as :
8	the spread of the disease within our provinces as a percentage of our total country infections is as follows :
9	test and production equipment
10	symptoms of the disease

### xho data: ###
1	Ingomso labantwana bakho lisezandleni zakho
2	* 500g yeesoseji ezincinci okanye iboerewors isikwe izilayi

In [6]:
from collections import Counter

char_freq_counters = {}

for lang, dataset in corpora.items():
    char_freq_counters[lang] = Counter()
    for text in dataset['train']['text']:
        char_freq_counters[lang].update(text)

print('ENG char total freq:', sum(char_freq_counters['eng'].values()))
print(char_freq_counters['eng'].most_common(10))
print(char_freq_counters['eng'].most_common()[-10:], end='\n\n')

print('XHO char total freq:', sum(char_freq_counters['xho'].values()))
print(char_freq_counters['xho'].most_common(10))
print(char_freq_counters['xho'].most_common()[-10:], end='\n\n')

print('AFR char total freq:', sum(char_freq_counters['afr'].values()))
print(char_freq_counters['afr'].most_common(10))
print(char_freq_counters['afr'].most_common()[-10:], end='\n\n')

ENG char total freq: 829219
[(' ', 137151), ('e', 79212), ('t', 61246), ('i', 57018), ('o', 52666), ('n', 51187), ('a', 50055), ('s', 43037), ('r', 42055), ('c', 26891)]
[('V', 7), ('°', 5), ('ó', 4), ('\ufeff', 3), ('é', 3), ('Y', 2), ('Ë', 2), ('í', 2), ('º', 2), ('K', 1)]

XHO char total freq: 912097
[(' ', 108939), ('a', 97015), ('e', 74994), ('i', 68376), ('n', 64294), ('u', 52916), ('o', 50766), ('k', 46873), ('l', 45863), ('h', 30787)]
[(';', 253), ('J', 215), ('/', 208), ('!', 195), ('V', 194), ('Q', 135), ('%', 75), ('é', 7), ('_', 2), ('á', 1)]

AFR char total freq: 840295
[(' ', 128853), ('e', 111261), ('i', 61786), ('n', 58866), ('a', 52751), ('r', 45823), ('o', 43265), ('s', 41035), ('t', 39205), ('d', 35937)]
[('ö', 12), ('±', 12), (']', 11), ('°', 5), ('Y', 3), ('J', 3), ('\ufeff', 3), ('Ê', 3), ('º', 2), ('Ï', 2)]



## 2. Preprocessing the data

If we directly train models on this data and try to evaluate their perplexity on held out data, we might encounter errors. That's because there may be rare characters that we don't see in training. (This would be a much bigger problem if our model was over words, but can be a problem even for characters.) We want our code to generalize to unseen characters, so we can still assign probabilities to sequences that contain them.

We will take a standard approach and replace all rare and unseen characters with a shared unknown character: `�` (the Unicode character for an unknown character). This allows our model to assign some probability to unseen characters and also prevents it overfitting to rare characters which are coincidentally only present in one of the corpora.

👉 **THINK:** What does this imply about the probabilities of different rare or unseen characters? How is it similar to add-alpha smoothing?

> 서로 다른 rare / unseen 문자들이 동일한 확률을 부여받음
> - Add-alpha smoothing과 유사: 모든 unseen events에 동일 확률 할당

Some languages do contain characters that don't occur in others! For instance, English doesn't use `क`, `ã` or `角`.

👉 **THINK:** Do you think an English n-gram model *should* assign zero probability to strings containing these characters? Why or why not? Can you think of any contexts where they might appear in English text?

> **Unseen ≠ Impossible**: ex. 외국어 인용문, 해당 언어 관련 기사 등에서 등장 가능

In order to remove rare characters, we need to decide what counts as "rare". Normally, you would take a look at your data statistics and perhaps try a few different frequency thresholds (tuning on a development set).

In this case, we've done some of that for you already and we are just going to use a threshold of 500. But you should still sanity-check the results by looking at what is getting removed.

👉 Run the next two cells to identify the rare characters and print them out.

In [7]:
from collections import Counter

def find_rare_chars(corpora, threshold=500):
    """ find characters that occur fewer than 'threshold' times across all corpora
    and return them as an alphanumerically sorted string """

    # count character frequencies in each corpus
    counters  = []
    for _, corpus in corpora.items():
        counts = Counter()
        for text in corpus['train']:
            counts.update(text['text'])
        counters.append(counts)

    all_chars = set([char for counter in counters for char in counter.keys()])

    # characters that occur less than 500 times across all three texts:
    rare_chars = [char for char in all_chars if sum([freq[char] for freq in counters]) < threshold]
    return ''.join(sorted(rare_chars)) # sorting alphnumerically to make output a bit easier to scan

In [8]:
rare_chars = find_rare_chars(corpora)
print(rare_chars)

!%*FHJQVWXYZ[]_°±µºÊËÏáèéêíóö–﻿


👉 **THINK:** Look at the rare characters. Do you think any of these characters shouldn't be replaced with the unknown character? If so, why not?

> 일반화(generalization)를 위해 모두 `�` 치환하는 것이 합리적
> - 일부 rare char는 언어별 특성(language-specific patterns)을 나타내는 단서일 수 있으나
> - 해당 말뭉치의 특정 분포에만 과적합(overfitting)할 위험이 있으므로

👉 **CHALLENGE QUESTION:** We know that encountering unseen characters in test data will cause problems. But why don't we simply deal with that at test time, by replacing the unseen characters with `�`? That is, why did we use `�` to replace characters that *did* occur in training, but rarely? (There are actually a few reasons, some more subtle than others. Don't spend more than a couple of minutes thinking about this now, you can come back to it later.)

> 1) `�` 발생 확률 (probability mass) 미리 학습 가능
> 2) rare token 발생 문맥 미리 학습 가능

👉 Now, run the next cell to actually replace the rare characters with the unknown character using a regular expression. (You'll learn more about these in CPSLP if you're taking it!)

In [9]:
import re
corpora = {
	lang: corpus.map(
		lambda x: {'text': re.sub(f"[{re.escape(rare_chars)}]", '�', x['text'])},
		# This regular expression identifies any character in rare_chars so we can replace it with the unknown character
	)
	for lang, corpus in corpora.items()
}

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [10]:
for lang, dataset in corpora.items():
    for split in ['train', 'test']:
        for text in dataset[split]['text']:
            if '�' in text:
                print(lang, split, text, sep='\t')
                break

eng	train	( 3 ) products containing chemicals listed in annexure b may be exported to countries that are not states parties to the chemical weapons convention : provided that the said products contain �
eng	test	any institution funded from the National Revenue �und or a Provincial Revenue �und or by a municipality ; 
xho	train	� 500g yeesoseji ezincinci okanye iboerewors isikwe izilayi eziyi-5cm
xho	test	INkundla �amatyala Esondo
afr	train	Bekragtiging van �etsontwerpe
afr	test	be�digde verklaring


## 3. Defining the n-gram model and looking at test cases

Now that we've preprocessed our corpora, it's time to build our n-gram model. We've written it so that our model works for different values of $N$. It also implements add-alpha smoothing.

👉 **THINK:** Why do we need smoothing, even after replacing unknown characters?

> ***unseen sequences*** (n-gram) 처리를 위해 여전히 필요

The code for the n-gram model is below. It's probably not a good use of the lab session time to go through every line right now, but we encourage you to review it in more detail later.

👉 For now, just make sure you understand the role of each function at a high level. Then, run the code.

In [11]:
import math
from collections import defaultdict, Counter

class CharNGramLM:
    def __init__(self, N=3, alpha=0.01):
        self.N = N
        if alpha < 0:
            raise ValueError("Invalid value of alpha!")
        else:
            self.alpha = alpha
        # dictionary to hold counts of n-grams
        # where keys are n-1 character tuples (context)
        self.context_counts = defaultdict(
            Counter
        )  # a defautltdict allows us to create a new Counter for each new n-1-gram automatically
        if N > 1:
            self.vocab = set(["<s>", "</s>"])
        elif N == 1:
            self.vocab = set(["</s>"])  # don't need BOS
        else:
            raise ValueError("Invalid value of N!")

    def train(self, corpus):
        ''' Given a corpus of sentences, store the counts of all character N-grams in the corpus.'''
        for sentence in corpus:
            # add start and end tokens
            sentence = ["<s>"] * (self.N - 1) + list(sentence) + ["</s>"]
            # update the counts of each n-gram
            for i in range(len(sentence) - self.N + 1):
                context = tuple(sentence[i : i + self.N - 1])
                char = sentence[i + self.N - 1]
                self.context_counts[context][char] += 1
                self.vocab.add(char)

    def print_counts(self):
        ''' Print out the counts of all N-grams that have non-zero counts, alphnumerically ordered.'''
        for context, counts in sorted(self.context_counts.items()):
            print(f"Context {context}:")
            for char, count in sorted(counts.items()):
                print(f"   C({char!r} | {context}) = {count}")

    def print_probs(self):
        ''' For each context in alphanumeric order, print out the conditional probability
        of each character in the vocabulary (including zero probabilities).'''
        for context, counts in sorted(self.context_counts.items()):
            print(f"Context {context}:")
            for char in sorted(self.vocab):
                if char != "<s>" : # We never generate <s> following another character, so it's not in the conditional probabilities
                    prob = self.prob(context, char)
                    print(f"   P({char!r} | {context}) = {prob}")

    # We've included the next two functions so it's easier for you to check
    # correctness on tiny examples, but we don't normally use raw
    # probabilities in models because they can underflow for very long
    # sequences. In this case, we do the computation in log space to
    # make sure it will always be right, and this function should also
    # work for reasonable sequence lengths.
    def prob(self, context, char):
        '''Returns the smoothed probability of char in the given context'''
        return 2 ** self.logprob(context, char)

    def prob_seq(self, sentence):
        '''Returns the  probability of the sentence, according to the model'''
        return 2 ** self.logprob_seq(sentence)

    def logprob(self, context, char):
        '''Returns the smoothed log probability of char in the given context'''
        context = tuple(context)
        counts = self.context_counts[context]
        if self.N == 1:
            V = len(self.vocab)
        else:
            V = len(self.vocab) - 1 # Only count characters we might generate next, which doesn't include <s>
        if (self.alpha == 0) and (counts[char] == 0):
            return -math.inf # Negative infinity
        else:
            prob = (counts[char] + self.alpha) / (sum(counts.values()) + self.alpha * V)
            return math.log2(prob)

    def logprob_seq(self, sentence):
        '''Returns the log probability of the sentence, according to the model'''
        sentence = ["<s>"] * (self.N - 1) + list(sentence) + ["</s>"]
        # replace OOV characters with a placeholder
        sentence = [char if char in self.vocab else "�" for char in sentence]
        score = 0.0
        for i in range(len(sentence) - self.N + 1):
            context = tuple(sentence[i : i + self.N - 1])
            char = sentence[i + self.N - 1]
            score += self.logprob(context, char)  # add the log probs of each n-gram (multiply the raw probabilities)
        return score

    def perplexity(self, corpus):
        '''Returns the perplexity of the model on the given corpus'''
        length = 0
        log_prob_sum = 0.0
        for sentence in corpus:
            log_prob_sum += self.logprob_seq(sentence)
            length += len(sentence) + 1  # account for </s>
        return 2 ** (-log_prob_sum / length)

Never assume that code is correct without testing it, whether it was written by you, us, or someone else (including AI)!

Before you train the model on the real data, you should check that it works correctly on a very small test case where you can check the results by hand. This will also help ensure that *you* understand how to compute the right result.

(It's not always possible to check results this way! Sometimes you'll need to think of other ways to test code, and ideally you can write automated test cases.)

We've constructed one such test case below.

👉 Before you run this test case, compute the counts, probabilities, and perplexity by hand. Remember that for a sequence of length $L$, the perplexity is $2^{(-1/L)*log P(seq)}$. You'll need to consider how the begin/end of sequence markers figure into the computations!

👉 Now run the test code below to check that your answers match the output of our model. If they don't, is there a bug in the model or in your own understanding?

👉 We started by checking the unigram model without smoothing, because it's the simplest, but it's important to ensure the code also works for other cases. Again **by hand**,
1. Figure out what counts and probabilities you should you get from this corpus if you use a bigram model (still with alpha = 0).
2. Do you expect the training and testing perplexities to be higher or lower than with the unigram model? Why?

👉 Now set N=2 and re-run the test code to check your answers. Do you see where smoothing becomes necessary? Update the value of alpha, rerun the code, and check that this change does what you expect.

In [12]:
# Tiny toy corpus
tiny_train = ['hi', 'ha', 'hi']
tiny_test = ['hi','i']

# 1. Unigram model without smoothing
tiny_lm = CharNGramLM(N=1,alpha=0)
tiny_lm.train(tiny_train)

tiny_lm.print_counts()
print()
tiny_lm.print_probs()
print()

print(tiny_lm.prob_seq('hi'))
print()

print (f"PPL on train: {tiny_lm.perplexity(tiny_train):.4}")
print (f"PPL on test: {tiny_lm.perplexity(tiny_test):.4}")

Context ():
   C('</s>' | ()) = 3
   C('a' | ()) = 1
   C('h' | ()) = 3
   C('i' | ()) = 2

Context ():
   P('</s>' | ()) = 0.3333333333333333
   P('a' | ()) = 0.11111111111111109
   P('h' | ()) = 0.3333333333333333
   P('i' | ()) = 0.22222222222222218

0.02469135802469135

PPL on train: 3.709
PPL on test: 3.528


In [13]:
# 2. Bigram model without smoothing
tiny_lm_bi = CharNGramLM(N=2,alpha=0)
tiny_lm_bi.train(tiny_train)

tiny_lm_bi.print_counts()
print()
tiny_lm_bi.print_probs()
print()

print(tiny_lm_bi.prob_seq('hi'))
print()

print (f"PPL on train: {tiny_lm_bi.perplexity(tiny_train):.6}")  # lower PPL
print (f"PPL on test: {tiny_lm_bi.perplexity(tiny_test):.6}")  # infinite PPL

Context ('<s>',):
   C('h' | ('<s>',)) = 3
Context ('a',):
   C('</s>' | ('a',)) = 1
Context ('h',):
   C('a' | ('h',)) = 1
   C('i' | ('h',)) = 2
Context ('i',):
   C('</s>' | ('i',)) = 2

Context ('<s>',):
   P('</s>' | ('<s>',)) = 0.0
   P('a' | ('<s>',)) = 0.0
   P('h' | ('<s>',)) = 1.0
   P('i' | ('<s>',)) = 0.0
Context ('a',):
   P('</s>' | ('a',)) = 1.0
   P('a' | ('a',)) = 0.0
   P('h' | ('a',)) = 0.0
   P('i' | ('a',)) = 0.0
Context ('h',):
   P('</s>' | ('h',)) = 0.0
   P('a' | ('h',)) = 0.3333333333333333
   P('h' | ('h',)) = 0.0
   P('i' | ('h',)) = 0.6666666666666666
Context ('i',):
   P('</s>' | ('i',)) = 1.0
   P('a' | ('i',)) = 0.0
   P('h' | ('i',)) = 0.0
   P('i' | ('i',)) = 0.0

0.6666666666666666

PPL on train: 1.23636
PPL on test: inf


In [14]:
# 3. Bigram model with add-1 smoothing
tiny_lm_bi_smth1 = CharNGramLM(N=2,alpha=1)
tiny_lm_bi_smth1.train(tiny_train)

# (C(w_i, w_i-1) + 1) / (C(w_i-1) + V)
# 이때 "V"는 4임 주의! --> {'h', 'i', 'a', '</s>'}
tiny_lm_bi_smth1.print_probs()
print()

print (f"PPL on train: {tiny_lm_bi_smth1.perplexity(tiny_train):.6}")  # higher PPL
print (f"PPL on test: {tiny_lm_bi_smth1.perplexity(tiny_test):.6}")  # non-infinite PPL

Context ('<s>',):
   P('</s>' | ('<s>',)) = 0.14285714285714285
   P('a' | ('<s>',)) = 0.14285714285714285
   P('h' | ('<s>',)) = 0.5714285714285714
   P('i' | ('<s>',)) = 0.14285714285714285
Context ('a',):
   P('</s>' | ('a',)) = 0.4
   P('a' | ('a',)) = 0.2
   P('h' | ('a',)) = 0.2
   P('i' | ('a',)) = 0.2
Context ('h',):
   P('</s>' | ('h',)) = 0.14285714285714285
   P('a' | ('h',)) = 0.2857142857142857
   P('h' | ('h',)) = 0.14285714285714285
   P('i' | ('h',)) = 0.42857142857142855
Context ('i',):
   P('</s>' | ('i',)) = 0.5
   P('a' | ('i',)) = 0.16666666666666669
   P('h' | ('i',)) = 0.16666666666666669
   P('i' | ('i',)) = 0.16666666666666669

PPL on train: 2.15948
PPL on test: 2.58009


## 4. Training on real corpora

Character n-gram models can be used for language identification by training models on different languages. Then, when you get a piece of text, you check its perplexity under each model. Whichever model has the lowest perplexity is identified as the language of the text.

Systems based on this idea are efficient and often work well, but they can still make errors!

Let's take a look at what kinds of examples might present problems for this sort of model, using a simple trigram model with default smoothing.

👉 Run the two cells below to train models on each of our three languages and compute the perplexities of some test sentences that could occur in English. Is the perplexity always lowest for the English model? If not, what sorts of input seem to cause problems for this way of doing language ID?

Feel free to add your own test sentences to the list to explore this question further.

> - 영어 모델이 항상 가장 낮은 Perplexity를 보이는 것은 아님.
> - **짧은 문장:** 정보량이 적어 Perplexity가 높고 변동성이 커지므로 언어 식별이 어려움.
> - **숫자 및 특이한 철자:** 언어별 특징적인 문자 패턴이 부족하거나 학습 데이터에 드물게 등장하여 오분류할 수 있음.
> - **소셜 미디어 텍스트:** 정제된 문서로 학습한 모델은 비표준 철자, 해시태그, 구어체 등이 포함된 텍스트에 취약함.

In [15]:
# Train a character n-gram language model for each language.
lms = {}
for lang, corpus in corpora.items():
    lm = CharNGramLM(N=3)
    lm.train(corpus['train']['text'])
    lms[lang] = lm

In [16]:
lms

{'eng': <__main__.CharNGramLM at 0x7e47c970a3f0>,
 'xho': <__main__.CharNGramLM at 0x7e47c9709f30>,
 'afr': <__main__.CharNGramLM at 0x7e47cb95ef90>}

In [17]:
my_test_sentences = ['I love natural language processing.',
                     'An incomplete sentence.',
                     'marketing and sales operations',
                     'Pierre Vinken, chairman of Elsevier, is well-known in NLP for appearing in the first sentence of the WSJ corpus.',
                     'Hi',
                     'no',
                     'See?',
                     'Aha, Lycketoft.', # Lycketoft was one of the words in English Europarl with count=1
                     '3601',
                     'hey @sloppyjoe wassup #chillin #fridaynight']

for sentence in my_test_sentences:
    print(f"Sentence: {sentence}")
    for lang, lm in lms.items():
        print(f"  {lang} ppl: {lm.perplexity([sentence]):.4f}")

Sentence: I love natural language processing.
  eng ppl: 15.8408
  xho ppl: 44.4424
  afr ppl: 16.9275
Sentence: An incomplete sentence.
  eng ppl: 16.1284
  xho ppl: 32.0180
  afr ppl: 20.6857
Sentence: marketing and sales operations
  eng ppl: 7.2362
  xho ppl: 40.1409
  afr ppl: 9.7060
Sentence: Pierre Vinken, chairman of Elsevier, is well-known in NLP for appearing in the first sentence of the WSJ corpus.
  eng ppl: 22.3198
  xho ppl: 53.7162
  afr ppl: 33.0819
Sentence: Hi
  eng ppl: 296.7452
  xho ppl: 124.1345
  afr ppl: 141.1031
Sentence: no
  eng ppl: 28.4848
  xho ppl: 102.5950
  afr ppl: 360.9987
Sentence: See?
  eng ppl: 456.0393
  xho ppl: 175.4139
  afr ppl: 644.3720
Sentence: Aha, Lycketoft.
  eng ppl: 358.2374
  xho ppl: 280.9362
  afr ppl: 263.4792
Sentence: 3601
  eng ppl: 254.9979
  xho ppl: 31.6438
  afr ppl: 238.0714
Sentence: hey @sloppyjoe wassup #chillin #fridaynight
  eng ppl: 103.6623
  xho ppl: 143.7036
  afr ppl: 66.4283


## 5. Generating from language models

This section is a small digression from the language ID task, to help you build intuitions about *generating* from language models.

**If you are running short on time** and want to focus on language ID, you can skip this section and come back to it later. Before going to Section 4, you will need to **run the first two cells below.**

Below, we've provided some code that implements several different generation (decoding) strategies. The default is just to sample from the language model probabilities (standard generation), but it also implements top-$k$ and temperature-scaled sampling. We will only explore top-$k$ sampling here, but if you want you can look at temperature-scaled sampling on your own after the lab.

To better see how the output is affected by both $N$ and the sampling  method, we'll generate from models with different values of $N$.

👉 Run the next two cells to train English models with different values of $N$ and implement generation. Then scroll down to the next question.

In [18]:
# Train some models on English with different N
eng_lms = {}
for N in [1, 3, 5, 10]:
    lm = CharNGramLM(N=N)
    lm.train(corpora['eng']['train']['text'])
    eng_lms[N] = lm

In [19]:
import random
def generate(model, top_k=0, temperature=1, max_len=100):
    '''Given a language model, use it to generate a single sequence.
    Generation will stop when </s> is generated, or after max_len chars (whichever comes first).
    If top_k = 0, we sample from the full distribution, otherwise re-normalize the top k choices and sample from those.
    '''
    context = ['<s>'] * (model.N - 1)
    result = []
    for _ in range(max_len):
        nlog_probs = []
        chars = []
        for c in model.vocab:
            log_p = model.logprob(context, c) / temperature
            nlog_probs.append(-log_p)
            chars.append(c)

        if top_k > 0:
            pairs = sorted(zip(chars, nlog_probs), key=lambda x: x[1])[:top_k]
            chars, nlog_probs = zip(*pairs)
            probs = [math.exp(-log_p) for log_p in nlog_probs]
        else:
            exp_probs = [math.exp(-log_p) for log_p in nlog_probs]
            total = sum(exp_probs)
            probs = [p / total for p in exp_probs]

        next_char = random.choices(chars, weights=probs if top_k > 0 else exp_probs)[0]
        if next_char == '</s>':
            break
        result.append(next_char)
        context = context[1:] + [next_char]
    return ''.join(result)

👉 The following cell is going to generate 10 sequences (up to 100 characters each) from each of the four English $N$-gram models, which have $N$=\[1, 3, 5, 10\].
1. Roughly speaking, what do you think the difference will be between the output of the four models? Run the code to find out if you are correct.
2. (**Challenge question**) Now take a closer look at the output. Do you see any places where the 5-gram or 10-gram model suddenly started generating total junk? (If not, you might need to rerun the code to get some new sequences. You should see this eventually.) Can you explain why that happened?
3. So far we used $k=0$, which actually turns off top-k sampling (so, we sample from the full distribution when generating each character). What do you think will happen if you change $k$ to be 1, and how might that change the output of each model? (*Hint*: What character will be chosen at each generation step?) Try it to see if you're right!
4. If you want, you can try other values of $k$ to see whether you can find a good balance between generating diverse outputs and not generating junk.

> 참조 문맥이 길수록 (N ↑) 실제 영어와 유사한 텍스트 생성.
> - Q2. 고차수 모델(5-gram, 10-gram)의 쓰레기(Junk) 텍스트 생성 원인?
    - Add-alpha smoothing은 학습 데이터에 등장하지 않은 문자 조합에도 0이 아닌 확률을 부여함.
    - 낮은 확률의 문자가 생성되면 기존에 거의 관찰되지 않은 문맥이 만들어지고, 모델은 다시 smoothing에 의존하게 됨.
    - 이러한 과정이 반복되면서 비정상적인 문자 생성이 연쇄적으로 발생하고, 결국 의미 없는 문자열(junk)이 생성될 수 있음.
    - 긴 N-gram 모델일수록 문맥이 구체적이고 데이터 희소성(data sparsity)이 심해 이러한 현상에 취약함.

In [20]:
k = 0
for N, lm in eng_lms.items():
    print("******* Generating with N =", N, ", k =", k, "*******")
    for i in range(1, 11):
        print(i, generate(lm, top_k=k), sep='\t')
    print()

******* Generating with N = 1 , k = 0 *******
1	nsMmw3nglr.0Rl6s3g;(/IjIPMEwgmKwOG8C. 3ftr/p?eO/EU6m2dsb:f/mM-PPrCr6zBCg,lkfzw
2	 ,UazLOf0x4G.l-.30E?
3	a(w4:NdrGL''.RTMeM:�yx(Në3SaaEOxtz51i,;s0f)p,'i1A
4	 s8I:.RlëTL;q)j4pR,;N/u?l P'vKDP,1Awb-m4U,, e,D7z9gqMsGdtNpy3i1�mzMwih(moca2K:TIRrtC,cwxR09vltGO/43S
5	tGxsmG.1Aup.x qsrj61i0MR p2
6	hnA8?Gë;-M1hBenOzpn3bOb6Sdx93993I;Uto9aM
7	 44Is)a96y,mdNOkiOcD Mif
8	e,6
9	igI2I:.?kaysUm?meR�h5Rë,pr3s?'jvsr8Obgp7vnav50ECffLNs7-n:?nëqU624UlC(g./ëëTu5AD-Bi37a;Lrc0itfT1k Rr3
10	tqUhO8p

******* Generating with N = 3 , k = 0 *******
1	ther vers thical red ) . 
2	a cond ints a se vintin of preasell of the to withe that in by or .
3	forifies dor per of the equent the mand peres .
4	( gove de , to ding appore se for govid bete hory the not accon sentegiver prommentagestay the mon t
5	sided ing the mund ints , a mayst the aph and for for scrity con ang accution of the reenterinces th
6	th th the ot spermition paccommuste thems , pord for .
7	tes the musted 

In [21]:
k = 1  # Greedy Decoding
for N, lm in eng_lms.items():
    print("******* Generating with N =", N, ", k =", k, "*******")
    for i in range(1, 11):
        print(i, generate(lm, top_k=k), sep='\t')
    print()

******* Generating with N = 1 , k = 1 *******
1	 ???????????????????????????????????????????????????????????????????????????????????????????????????
2	 ???????????????????????????????????????????????????????????????????????????????????????????????????
3	 ???????????????????????????????????????????????????????????????????????????????????????????????????
4	 ???????????????????????????????????????????????????????????????????????????????????????????????????
5	 ???????????????????????????????????????????????????????????????????????????????????????????????????
6	 ???????????????????????????????????????????????????????????????????????????????????????????????????
7	 ???????????????????????????????????????????????????????????????????????????????????????????????????
8	 ???????????????????????????????????????????????????????????????????????????????????????????????????
9	 ???????????????????????????????????????????????????????????????????????????????????????????????????
10	 ???????????????????????

In [22]:
k = 5
for N, lm in eng_lms.items():
    print("******* Generating with N =", N, ", k =", k, "*******")
    for i in range(1, 11):
        print(i, generate(lm, top_k=k), sep='\t')
    print()

******* Generating with N = 1 , k = 5 *******
1	 ww7Ewywww7w?wwwwy?7w?7Ew7wEyywww7EyyE?Ew7?7y77Ewy7?w?wwwyEyw??wy?Ey7?y??ywyw7w77yyEw???wy7EywwwE?Ey
2	eEEEy?Ey?yy?EEEw7wwwEyE?EE7?w77yEy?yy7E7yywwEE7?y??Eyw7EE?yEy7y7Ew???y?wE7?E7?7wywwE7?w?7?wwy?wywEEw
3	e??77wwyyy??yw7EE7EwyyEEw7??7?ywy7wEEw??EEwEwwyEyE7w?77wE?7w7E7?wyEEEyEEyyyy??y?EwwEwE77wyy7w?wEywyw
4	 w7E??yE7?yw?y?wEwyy?wEw7w7?7EEwwEyww7ww7E7yy?EE7EwwEwy7w??y7wyyy?77ww777E7?ww7y7?yyw7Ey7Eyw?y7yEwyE
5	o?yy?7EEy?7ywywwyyy7E?EE??7E?wy?7?E?ww?yE77ww??77E7E77yE7yyyy77ywyE?7E7?EwEEy?Eyyywyw7y?yE?wEy??Eyww
6	ty77Eyy?wEwEEyE?wy?7EE?7Eyy7wy??y??7w7EwEy7wyEEyE7yEy???E?w7?wEwywyE7EEEE??wwE?yw7yEyE7yyw????w7Eyyy
7	e?77y???wyy7y7EwEywy7E?EwwyyE7E?wy7ywwEE7Eyw7?E??E777wy?wEEyEyw7?EywEE7w7y?y7?E?77yy777E?EyE7Ewy7wy7
8	t7EwE??7y?E?wwE7w?Ey7E?yy?y?EyEw?7ywEywEw?7EEwE7ywE?ywy?yEyEy7wyEEy??7yw?yyE7ywEE?EE7yw7?yEwy7?w?wEy
9	iy7?wE7?w?w?7?Ewy??ywy77wEy?7E7w?E?ywE?wy7?7?E7EyEEwyw?wwwyEww7wEyE?EwEEywywyEEw7ywyyEw77EywyE7EEy?7
10	ewEw?yyw7777?wwy7777E7EE

👉 **THINK:** If we want to get the best performance from our language ID system, we will need to tune the model hyperparameters (as we'll do in the next section). Should we include $k$ in the hyperparameters that we tune? Why or why not?

> - k는 텍스트 생성(generation) 과정에서만 영향을 미치는 파라미터임.
> - Language ID는 입력 문장의 확률을 계산하여 언어를 판별(scoring)하므로 k와 무관함.
> - 따라서 Language ID의 하이퍼파라미터 튜닝 대상에 k를 포함할 필요가 없음.
> - 단, Dialogue System처럼 실제 텍스트를 생성하는 시스템에서는 k를 튜닝할 수 있음.

## 6. Exploring the effects of $\alpha$

Now, let's see how the smoothing parameter affects the generated text and the model perplexity.

For generation, we  will use regular sampling ($k=0$). Our code uses a 5-gram model, but you can try other values of $N$ later if you want.

Notice that we are exploring $\alpha$ values over several orders of magnitude. This is a common pattern in NLP and machine learning, especially if you don't have a good idea of the range that will work well. If you find huge differences between the results, you can always try more values in between the big jumps.

👉 Run the code and look at the generated output.
1. How does the value of alpha affect the generation quality?
2. Will the alpha that yields the best quality output also have the lowest perplexity on the development set? Why or why not?
3. Now fill in the code to actually compute the perplexities on the training and development parts of the English data. Was your prediction correct?

> 1.
> - Alpha가 클수록 학습 데이터의 실제 N-gram 분포에서 벗어나므로 생성 품질이 나빠짐.
> - Alpha가 작을수록 학습 데이터의 분포를 잘 반영하여 자연스러운 텍스트가 생성됨.

> 2.
> - 생성 품질이 가장 좋은 Alpha가 Development Perplexity까지 가장 낮게 만드는 것은 아님.
> - Alpha = 0이면 학습 데이터에 없는 N-gram의 확률이 0이 되어 Development Perplexity가 무한대가 될 수 있음.
> - 따라서 적절한 Smoothing이 필요함.

> 3.
> - Training Perplexity: Alpha가 작아질수록 낮아짐.
> - Development Perplexity: Alpha = 0.001에서 가장 낮으며, 그보다 작아지면 다시 증가함.

In [23]:
N = 5
lm = eng_lms[N]
for i, alpha in enumerate([1, .1, .01, .001, .0001, 1e-5, 0], 1):
    lm.alpha = alpha
    print(f"******* ({i}) Model with N =", N, ", alpha =", alpha, "*******")
    train_ppl = lm.perplexity(corpora['eng']['train']['text'])
    dev_ppl = lm.perplexity(corpora['eng']['test']['text'])
    print(f"Perplexities: {train_ppl:.5} (train), {dev_ppl:.5} (dev)", end='\n\n')
    print("** Generated text: **")
    for j in range(1, 11):
        print(j, generate(lm), sep='\t')
    print('\n')

******* (1) Model with N = 5 , alpha = 1 *******
Perplexities: 4.2671 (train), 4.5268 (dev)

** Generated text: **
1	the apri3w08SvEynDëUzhqM0:pdT<s>nR;br1)A-GCOBvwev3/8A d1mSrb8MKh/c:fUPx)lnLRbeMi04d6leig3t5AlB'h874Oe7
2	or the parties as cbs)ë1/rt<s>z 348gCf,;LzINq(UeE0Bed�<s>pLu1spt9<s>c9,OT<s>5EMf2hNtAKf66C5P'gaMO1P1UTdzw1Bs
3	any sbdBAuD/lTLucbëkI,xdwO(b4etTaëf/tIU-9k,2iONj'2-G3IUlIu
4	man3r5I P1a.xao<s>9Up1bGpL:
5	technical governments h9 Gw.lSuy?m0RiD/.?i7xTU52v22T7Pk9RhSPU8aoU
6	and devic�hTh Na<s>-k.'qh'-5coS,05KimPaf7SDoxë?i
7	SubjPMS8O<s>-3ORS?vqëA?fyySEp9bvjhIA'3AKz<s>
8	i wish ao;pmEUvGb�rN'g,UEveyR3u7DcP3wD
9	and in the performance within a mixe'ë n7wzjn5S(M9(Ljvmljn
10	( 1 ) and oR:C?AëpDB8:9z�15f/ftO aA,;'O')eLUSOcejvuOqn:k:d6C8B.ls<s>Dn/;,cR)o


******* (2) Model with N = 5 , alpha = 0.1 *******
Perplexities: 2.8295 (train), 3.0531 (dev)

** Generated text: **
1	the new control systems spections and
2	the bill be executive and paid ?
3	an ' is executive , the vehicles 

👉 **THINK:** We found the best value of $\alpha$ for a particular value of $N$, on the English data. Do you think the same value of $\alpha$ will be best for other values of $N$?  What about on the other languages?

## 7. Optimizing the hyperparameters

Now let's be more systematic about finding the best values for $N$ and $\alpha$. We will use a *grid search*, which just means we will loop over all possible pairs of values and pick the hyperparameters that have the lowest perplexity on the development set.

Grid search is simple to implement, but can be very inefficient if the model needs to be re-trained for many hyperparameters or many possible values of each.

In this case, we only need to train models for each value of $N$, because we can change the model's smoothing parameter without re-training. (This might not be true for all smoothing methods or all implementations of add-alpha smoothing, but it is true for ours!)

We won't explore every possible option for $N$, but will look at a good range.

👉 Run the next cell to find the hyperparameters that work best on our dev sets.

In [24]:
def tune_hyperps(lang, corpus, verbose = False):
    '''Tunes the hyperparameters N and alpha by doing a grid search,
    training on the train portion of corpus, and choosing the N and alpha
    that have the lowest perplexity on the dev portion of corpus.
    lang is a string identifying the  language of corpus.
    If verbose = True, prints out perplexities of all models tested.'''

    best_alpha = None
    best_lm = None
    best_n = None
    best_prpl = float('inf')

    for N in [3, 5, 8, 10, 15]:
        lm = CharNGramLM(N=N)
        lm.train(corpus['train']['text'])
        print(f"Searching alphas with N = {N} for {lang}")

        for alpha in [1, 0.1, 0.01, 0.001, 0.0001, 1e-05]:
            lm.alpha = alpha
            prpl = lm.perplexity(corpus['test']['text'])
            if verbose: print(f'{alpha} {prpl}')
            if prpl < best_prpl:
                best_prpl = prpl
                best_alpha = alpha
                best_n = N
                best_lm = lm # gives a pointer to this lm (not a copy)
    print(f"Best (N, alpha) for {lang}: ({best_n}, {best_alpha}) with perplexity: {best_prpl}", end='\n\n')
    best_lm.alpha = best_alpha
    return best_lm

lms_tuned = {} # this will store the best models for each language
for lang, corpus in corpora.items():
    lms_tuned[lang] = tune_hyperps(lang, corpus, False)

Searching alphas with N = 3 for eng
Searching alphas with N = 5 for eng
Searching alphas with N = 8 for eng
Searching alphas with N = 10 for eng
Searching alphas with N = 15 for eng
Best (N, alpha) for eng: (10, 0.001) with perplexity: 1.8945048205481836

Searching alphas with N = 3 for xho
Searching alphas with N = 5 for xho
Searching alphas with N = 8 for xho
Searching alphas with N = 10 for xho
Searching alphas with N = 15 for xho
Best (N, alpha) for xho: (5, 0.01) with perplexity: 4.797152803245561

Searching alphas with N = 3 for afr
Searching alphas with N = 5 for afr
Searching alphas with N = 8 for afr
Searching alphas with N = 10 for afr
Searching alphas with N = 15 for afr
Best (N, alpha) for afr: (10, 0.001) with perplexity: 1.9597397404081898



Notice that the best models for English and Afrikaans are based on very long character strings (10-grams), and also have very low perplexity. (Remember that log(ppl) is roughly the number of bits of uncertainty per character, so on average each character has fewer than 1 bit of uncertainty --- less than a fair coin flip.)

👉 **THINK:** What might this tell you about the English and Afrikaans datasets, as compared to the Xhosa one?

> - English와 Afrikaans 모델은 10-gram에서 매우 낮은 Perplexity를 보임.
> - 이는 두 데이터셋이 Xhosa 데이터셋보다 반복적인 패턴이 많고, 학습 데이터와 Development 데이터의 분포가 더 유사하다는 것을 시사함.
> - 특히 적은 학습 데이터로도 10-gram 모델이 좋은 성능을 보이는 것은 다소 이례적이므로, 데이터셋에 문제가 있을 가능성을 의심할 필요가 있음.

## 8. Language identification

Now that we've tuned the hyperparameters, we are ready to try using our models for language identification! We will do this using the three files UDHR.1, UDHR.2, and UDHR.3, which contain the same text (the Universal Declaration of Human Rights) in each of the three languages. These are our *test sets*, since we are using them to see how well our models perform on data that we have not used for tuning.

👉 Run the code below to compute the perplexity of each document under each of the three optimized models, then consider the following questions:

1. According to the model perplexities, which language is each document most likely to be written in? Did the models identify the languages correctly? (You can open each document to check.)
2. For each model, compare its perplexity on the development set (from the previous section) to its perplexity on the test set from the same language. Do you notice anything surprising? If so, do you have any possible explanations for this surprising result?
3. These test documents are quite long, but many texts aren't. Given the results you've seen here and elsewhere in the lab, which of these models do you think is most likely to work well for language ID on short documents? Why?
4. Suppose you had more time to work on this problem. In light of your answers to the previous questions, do you have any thoughts about how you could either *check* whether your explanations are correct, or *improve* the models to work better for language ID?

1. **Language Identification 결과**
> - 각 문서의 Perplexity가 가장 낮은 모델을 해당 문서의 언어로 예측함.
> - UDHR.1 → Xhosa, UDHR.2 → Afrikaans, UDHR.3 → English
> - 세 문서 모두 언어를 올바르게 식별함.

2. **Development vs Test Perplexity**
> - Xhosa 모델은 Development와 Test Perplexity가 비슷하여 일반화 성능이 양호함.
> - English와 Afrikaans 모델은 Development보다 Test Perplexity가 크게 증가함.
> - 학습 및 Development 데이터의 높은 반복성으로 인한 Overfitting 가능성이 있음.
> - 상대적으로 낮은 차수의 N-gram 모델이 새로운 데이터에 더 잘 일반화할 수 있음을 시사함.

3. **Short Document에서의 성능**
> - 짧은 문서는 Perplexity의 변동성이 커서 언어 식별이 더 어려움.
> - English와 Afrikaans 모델은 Test Perplexity가 높고 언어 간 Perplexity 차이도 작으므로 짧은 문서에서 성능이 떨어질 가능성이 큼.
> - 상대적으로 일반화 성능이 좋은 Xhosa 모델이 더 유리할 것으로 예상됨.

4. **개선 및 검증 방법**
> - 중복된 문장이나 구절이 있는지 확인하고 제거하기.
> - 데이터를 무작위로 섞는 대신 원래 순서를 유지하여 Train/Dev 간 차이를 확보하는 방법 고려.
> - 동일한 장르의 다양한 출처에서 데이터를 수집하여 데이터 다양성 확보하기.
> - Test set에 맞춰 하이퍼파라미터를 튜닝하지 않고, 별도의 Development set으로 일반화 성능을 평가하기.

In [25]:
# test on the UDHR
for test_set in ['UDHR.1', 'UDHR.2', 'UDHR.3']:
    print(f"## Testing on {test_set} ##")

    with open(test_set) as f:
        udhr_text = f.readlines()
        udhr_text = [line.strip() for line in udhr_text if line.strip()] # remove empty lines

        for lang, lm in lms_tuned.items():
            lm = lms_tuned[lang]
            perplexity = lm.perplexity(udhr_text)
            print(f"Average perplexity for {lang} on {test_set}: {perplexity}")

    print()

## Testing on UDHR.1 ##
Average perplexity for eng on UDHR.1: 69.16492790879369
Average perplexity for xho on UDHR.1: 5.145040494705159
Average perplexity for afr on UDHR.1: 69.07042324821619

## Testing on UDHR.2 ##
Average perplexity for eng on UDHR.2: 64.96759542849284
Average perplexity for xho on UDHR.2: 93.52250064642094
Average perplexity for afr on UDHR.2: 25.55759847044102

## Testing on UDHR.3 ##
Average perplexity for eng on UDHR.3: 24.163496902148967
Average perplexity for xho on UDHR.3: 59.76816172430906
Average perplexity for afr on UDHR.3: 60.66358384759715



## &#127881; &#127881; Congratulations! You're done! (Or just getting started?) &#x1F680;

We hope you enjoyed exploring N-gram models and language ID!

We wanted to give you a sense for how choices about data can make a big difference to the performance of an NLP system, in ways that are not always obvious at the beginning. Developing a system is often a very iterative process.

Of course, now that you have found some problems with the setup we used here, and thought of ideas about how you could improve it, you might want to actually try some of those! Again, this is totally optional, but if you do any of that exploration and find something interesting, we'd love to hear about it on Piazza!